# Session 1 · Generative AI for Creators: Foundations and Toolkit

Generative AI › Track 3 · Innovate with Generative AI

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Runtime ▸ Run all in Colab). Outputs below were produced by actually executing this notebook.

### Example · `session-01_modality_map.py`

In [1]:
"""
Session 1 · Example 1 — Map a creative task to the right Gen AI modality.

Idea: before opening any tool, decide WHAT kind of output you need
(text, image, audio, video, slides, code). This script does that with
simple keyword rules and prints a starter prompt for each task.

Run:  python session-01_modality_map.py
Only the Python standard library is used.
"""
import re

# Each modality has trigger words and a starter prompt skeleton.
MODALITIES = {
    "text":   {"words": ["essay", "blog", "caption", "email", "report", "script", "post"],
               "tools": "chat assistants (ChatGPT, Gemini, Copilot, Claude)",
               "skeleton": "Act as a {role}. Write a {item} about {topic} for {audience}. Format: {fmt}."},
    "image":  {"words": ["poster", "logo", "illustration", "thumbnail", "photo", "banner"],
               "tools": "image generators (DALL-E, Firefly, Midjourney, Ideogram)",
               "skeleton": "{item} of {topic}, style: {style}, lighting: {light}, aspect ratio {ar}."},
    "audio":  {"words": ["podcast", "voiceover", "jingle", "narration", "music"],
               "tools": "voice and music tools (ElevenLabs, Suno, text-to-speech)",
               "skeleton": "Generate a {item} about {topic}, voice: {voice}, length: {length}."},
    "video":  {"words": ["reel", "video", "short", "trailer", "animation"],
               "tools": "video tools (Runway, Pika, Sora, InVideo, CapCut)",
               "skeleton": "A {length} {item} showing {topic}, camera: {camera}, mood: {mood}."},
    "slides": {"words": ["presentation", "deck", "slides", "pitch"],
               "tools": "slide tools (Gamma, Copilot in PowerPoint, Canva)",
               "skeleton": "Create a {n}-slide deck on {topic} for {audience}; one idea per slide."},
}


def classify(task: str) -> str:
    """Return the modality whose trigger word appears in the task (first match wins).

    We compare WHOLE words. A plain substring test would wrongly find "post"
    inside "poster" and send a poster job to a text tool.
    """
    task_words = set(re.findall(r"[a-z]+", task.lower()))
    for name, info in MODALITIES.items():
        if task_words & set(info["words"]):
            return name
    return "text"  # sensible default: most creative work starts as words


tasks = [
    "Instagram caption for a campus bake sale",
    "Poster for the Spring coding contest",
    "30-second voiceover for the department intro",
    "Reel showing a day in the CS lab",
    "Pitch deck for a student startup",
    "Thank-you note to guest speaker",  # no trigger word -> default
]

print(f"{'Task':45} | {'Modality':8} | Tool category")
print("-" * 100)
for t in tasks:
    m = classify(t)
    print(f"{t:45} | {m:8} | {MODALITIES[m]['tools']}")

print("\nStarter skeleton for the poster task:")
print(MODALITIES["image"]["skeleton"])

Task                                          | Modality | Tool category
----------------------------------------------------------------------------------------------------
Instagram caption for a campus bake sale      | text     | chat assistants (ChatGPT, Gemini, Copilot, Claude)
Poster for the Spring coding contest          | image    | image generators (DALL-E, Firefly, Midjourney, Ideogram)
30-second voiceover for the department intro  | audio    | voice and music tools (ElevenLabs, Suno, text-to-speech)
Reel showing a day in the CS lab              | video    | video tools (Runway, Pika, Sora, InVideo, CapCut)
Pitch deck for a student startup              | slides   | slide tools (Gamma, Copilot in PowerPoint, Canva)
Thank-you note to guest speaker               | text     | chat assistants (ChatGPT, Gemini, Copilot, Claude)

Starter skeleton for the poster task:
{item} of {topic}, style: {style}, lighting: {light}, aspect ratio {ar}.


### Example · `session-01_tiny_generator.py`

In [2]:
"""
Session 1 · Example 2 — A tiny "generative model" you can read in one minute.

A real large language model learns probabilities for the next token from
billions of words. Here we do the same thing at toy scale: count which word
follows which in a small corpus (a bigram model), then SAMPLE new sentences.

Key lesson: the model only recombines patterns it has seen. Changing the
random seed changes the output, just like re-generating in a chat tool.

Run:  python session-01_tiny_generator.py
"""
import random
from collections import defaultdict

corpus = """
the poster shows a bright city at night .
the poster uses bold orange text .
the video shows a calm city at sunrise .
the video uses soft music and slow camera moves .
the story follows a young coder at night .
the story uses short scenes and bold choices .
"""

# 1) Learn: for every word, remember all words that came right after it.
follows = defaultdict(list)
words = corpus.split()
for current, nxt in zip(words, words[1:]):
    follows[current].append(nxt)

print("What the model learned after 'the':", sorted(set(follows["the"])))
print("What the model learned after 'uses':", sorted(set(follows["uses"])))


# 2) Generate: start with a word and keep sampling a likely next word.
def generate(start: str, seed: int, max_words: int = 12) -> str:
    rng = random.Random(seed)       # fixed seed => reproducible output
    out = [start]
    while len(out) < max_words:
        options = follows.get(out[-1])
        if not options:
            break
        nxt = rng.choice(options)   # frequent pairs appear more often in the list
        out.append(nxt)
        if nxt == ".":
            break
    return " ".join(out)


print()
for seed in (1, 2, 3, 4):
    print(f"seed {seed}: {generate('the', seed)}")

What the model learned after 'the': ['poster', 'story', 'video']
What the model learned after 'uses': ['bold', 'short', 'soft']

seed 1: the poster shows a bright city at sunrise .
seed 2: the poster shows a calm city at sunrise .
seed 3: the poster shows a young coder at night .
seed 4: the poster uses bold choices .


### Example · `session-01_project_toolkit_planner.py`

In [3]:
"""
Session 1 · Mini project (full solution) — Creator Toolkit Planner.

Problem: a student society must deliver a launch pack for its annual tech
fest: a blog post, a poster, a 30-second promo reel, a narration track and a
5-slide sponsor deck. Budget: Rs 0 (free tiers only). Plan which tool category
to use, the estimated effort with and without AI, and the review step for each
deliverable, then draw a bar chart of the time saved.

Run:  python session-01_project_toolkit_planner.py
Creates: session-01_time_saved.png  (needs matplotlib: pip install matplotlib)
"""
from dataclasses import dataclass

import matplotlib
matplotlib.use("Agg")               # draw to a file, no window needed
import matplotlib.pyplot as plt


@dataclass
class Deliverable:
    name: str
    modality: str
    manual_min: int      # minutes if done fully by hand
    ai_min: int          # minutes WITH AI, including prompting
    review_min: int      # minutes a human spends checking the AI output
    review_step: str

    @property
    def total_ai(self) -> int:
        return self.ai_min + self.review_min

    @property
    def saved_pct(self) -> float:
        return 100 * (self.manual_min - self.total_ai) / self.manual_min


pack = [
    Deliverable("Blog post (600 words)", "text",   120, 25, 20, "Fact-check dates, names, sponsor list"),
    Deliverable("Event poster",          "image",  180, 30, 25, "Check spelling in the image, contrast, logo use"),
    Deliverable("30-sec promo reel",     "video",  300, 60, 40, "Watch at full speed; check captions and music licence"),
    Deliverable("Narration track",       "audio",   90, 15, 10, "Listen for mispronounced names"),
    Deliverable("5-slide sponsor deck",  "slides", 150, 30, 30, "Verify every number and the contact details"),
]

print(f"{'Deliverable':24}{'Modality':9}{'Manual':>8}{'AI+Review':>11}{'Saved':>8}  Review step")
print("-" * 110)
for d in pack:
    print(f"{d.name:24}{d.modality:9}{d.manual_min:>8}{d.total_ai:>11}{d.saved_pct:>7.0f}%  {d.review_step}")

manual = sum(d.manual_min for d in pack)
with_ai = sum(d.total_ai for d in pack)
review = sum(d.review_min for d in pack)
print("-" * 110)
print(f"Total manual: {manual} min ({manual/60:.1f} h) | with AI incl. review: {with_ai} min ({with_ai/60:.1f} h)")
print(f"Review share of AI time: {100*review/with_ai:.0f}%  -> reviewing is a real part of the job")

# Chart: grouped bars, manual vs AI+review
labels = [d.name.split(" (")[0] for d in pack]
x = range(len(pack))
plt.figure(figsize=(8, 4))
plt.bar([i - 0.2 for i in x], [d.manual_min for d in pack], width=0.4, label="Manual", color="#A78BFA")
plt.bar([i + 0.2 for i in x], [d.total_ai for d in pack], width=0.4, label="AI + review", color="#FF6B1A")
plt.xticks(list(x), labels, rotation=15)
plt.ylabel("Minutes")
plt.title("Launch pack: estimated minutes per deliverable")
plt.legend()
plt.tight_layout()
plt.savefig("session-01_time_saved.png", dpi=80)
print("Saved chart -> session-01_time_saved.png")

Deliverable             Modality   Manual  AI+Review   Saved  Review step
--------------------------------------------------------------------------------------------------------------
Blog post (600 words)   text          120         45     62%  Fact-check dates, names, sponsor list
Event poster            image         180         55     69%  Check spelling in the image, contrast, logo use
30-sec promo reel       video         300        100     67%  Watch at full speed; check captions and music licence
Narration track         audio          90         25     72%  Listen for mispronounced names
5-slide sponsor deck    slides        150         60     60%  Verify every number and the contact details
--------------------------------------------------------------------------------------------------------------
Total manual: 840 min (14.0 h) | with AI incl. review: 285 min (4.8 h)
Review share of AI time: 44%  -> reviewing is a real part of the job
Saved chart -> session-01_time_saved.pn